# A.1: Building labels with a loss mask

Render a two-turn conversation with SmolLM2's chat template and mark which tokens are trained.


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
IGNORE = -100

def build_example(messages, tok):
    """Token IDs and labels for a conversation; only assistant turns (and their
    end-of-turn token) are trained, everything else gets the label IGNORE."""
    ids, labels = [], []
    for k in range(len(messages)):
        # Render the conversation up to and including turn k, and keep the new tokens.
        text = tok.apply_chat_template(messages[:k + 1], tokenize=False)
        new = tok(text, add_special_tokens=False).input_ids[len(ids):]
        if messages[k]["role"] == "assistant":
            header = tok.apply_chat_template(messages[:k], tokenize=False,
                                             add_generation_prompt=True)
            n_header = len(tok(header, add_special_tokens=False).input_ids) - len(ids)
            # The assistant header ("<|im_start|>assistant\n") is prompt, the rest is trained,
            # except the final newline after <|im_end|>, which the template adds between turns.
            body = new[n_header:]
            lab = [IGNORE] * n_header + body[:-1] + [IGNORE]
        else:
            lab = [IGNORE] * len(new)
        ids += new
        labels += lab
    return ids, labels

conversation = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user", "content": "And of Japan?"},
    {"role": "assistant", "content": "Tokyo."},
]
ids, labels = build_example(conversation, tok)
assert tok.decode(ids) == tok.apply_chat_template(conversation, tokenize=False)
trained = [i for i, l in zip(ids, labels) if l != IGNORE]
print("tokens:", len(ids), "| trained:", len(trained))
print("trained tokens:", [tok.decode([i]) for i in trained])


**Expected output**

Output:

```text
tokens: 49 | trained: 7
trained tokens: ['Paris', '.', '<|im_end|>', 'To', 'kyo', '.', '<|im_end|>']
```
